In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.interpolate import griddata
import os


In [2]:
mesh_size = 2.0
normal_stress = 12.5  # 根據您的資料夾調整
Spring_stiffness = 3000

control_mode = 2  # Stress Control (DC mode)
CONTROL_MODES = ["NormalStressDC", "NormalStressPC", "NormalStressDC-New"]
CONTROL_MODES_TITLE = ["Displacement Control", "Stress Control", "Displacement Control (New B.C.)"]

DATA_FOLDER = f'./Disp-Ana/{normal_stress}MPa/ShearFace'  # 調整為您的路徑
RUPTURE_POSITIONS = [5, 15, 25, 35, 45, 55, 65, 75, 80, 85, 90, 95, 100, 105, 110, 115]

In [3]:
GIF_FOLDER = os.path.join(DATA_FOLDER, 'GIFS')
os.makedirs(GIF_FOLDER, exist_ok=True)

# ======== 儲存用容器 ========
all_u2_right_data, all_u2_center_data, all_u2_relative_data, all_positions = [], [], [], []

# ======== 讀取 npz ========
print("Collecting U2 displacement data for GIF generation...")
for pos in RUPTURE_POSITIONS:
    f = os.path.join(DATA_FOLDER, f'ShearFace-{pos}.npz')
    if not os.path.exists(f):
        print(f"File {f} not found, skipping...")
        continue

    d = np.load(f)
    if 'u2_right' not in d or 'u2_center' not in d:
        print(f"  No U2 displacement data in {f}")
        continue

    # 右側與中央資料
    y_r, z_r, u2_r = d['y_right'], d['z_right'], d['u2_right']
    y_c, z_c, u2_c = d['y_center'], d['z_center'], d['u2_center']
    grid_y, grid_z = 400, 200

    # ---------- RIGHT ----------
    if y_r.size:
        Y_r, Z_r = np.meshgrid(np.linspace(y_r.min(), y_r.max(), grid_y),
                               np.linspace(z_r.min(), z_r.max(), grid_z))
        U2_r = griddata(np.c_[y_r, z_r], u2_r, (Y_r, Z_r), method='linear')
        all_u2_right_data.append(dict(Y=Y_r, Z=Z_r, U=U2_r, pos=pos,
                                      y_range=(y_r.min(), y_r.max()),
                                      z_range=(z_r.min(), z_r.max())))

    # ---------- CENTER ----------
    if y_c.size:
        Y_c, Z_c = np.meshgrid(np.linspace(y_c.min(), y_c.max(), grid_y),
                               np.linspace(z_c.min(), z_c.max(), grid_z))
        U2_c = griddata(np.c_[y_c, z_c], u2_c, (Y_c, Z_c), method='linear')
        all_u2_center_data.append(dict(Y=Y_c, Z=Z_c, U=U2_c, pos=pos,
                                       y_range=(y_c.min(), y_c.max()),
                                       z_range=(z_c.min(), z_c.max())))

    # ---------- RELATIVE ----------
    if y_r.size and y_c.size:
        y_min, y_max = max(y_r.min(), y_c.min()), min(y_r.max(), y_c.max())
        z_min, z_max = max(z_r.min(), z_c.min()), min(z_r.max(), z_c.max())
        Y_com, Z_com = np.meshgrid(np.linspace(y_min, y_max, grid_y),
                                   np.linspace(z_min, z_max, grid_z))
        U2_r_com = griddata(np.c_[y_r, z_r], u2_r, (Y_com, Z_com), method='linear')
        U2_c_com = griddata(np.c_[y_c, z_c], u2_c, (Y_com, Z_com), method='linear')
        all_u2_relative_data.append(dict(Y=Y_com, Z=Z_com, U=U2_r_com - U2_c_com, pos=pos,
                                         y_range=(y_min, y_max), z_range=(z_min, z_max)))
    all_positions.append(pos)

# ======== 通用函式 ========
# def make_heatmap_gif(data_list, cmap, title_prefix, fname):
#     """右／中位移 GIF；每幀 ax.clear() 後重畫，避免舊 artist 留存。"""
#     if not data_list:
#         return
#     vals = np.concatenate([d['U'][~np.isnan(d['U'])] for d in data_list])
#     vmin, vmax = np.percentile(vals, [1, 99])

#     y_min = min(d['y_range'][0] for d in data_list)
#     y_max = max(d['y_range'][1] for d in data_list)
#     z_min = min(d['z_range'][0] for d in data_list)
#     z_max = max(d['z_range'][1] for d in data_list)

#     fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)

#     def animate(i):
#         d = data_list[i]
#         ax.clear()
#         im = ax.contourf(d['Y'], d['Z'], d['U'], 50, cmap=cmap,
#                          vmin=vmin, vmax=vmax)
#         ax.set(xlabel='Y (mm)', ylabel='Z (mm)',
#                title=f'{title_prefix} – {d["pos"]} mm\n'
#                      f'{normal_stress:.1f} MPa,  E={Spring_stiffness} MPa',
#                xlim=(y_min, y_max), ylim=(z_min, z_max), aspect='equal')
#         ax.grid(True, alpha=0.3)
#         cbar = fig.colorbar(im, ax=ax, label='Relative U2 (mm)')
#         return []                     # blit=False -> 不需回傳 artist

#     anim = FuncAnimation(fig, animate, frames=len(data_list), interval=500, blit=False)
#     path = os.path.join(GIF_FOLDER, fname)
#     anim.save(path, writer=PillowWriter(fps=2), dpi=150)
#     plt.close(fig)
#     print(f"{title_prefix} GIF saved: {path}")

# # ======== 產生右側／中央 GIF ========
# print("\nCreating U2 Right Side Heatmap GIF ...")
# make_heatmap_gif(all_u2_right_data, 'coolwarm',
#                  'U2 Displacement (Right Side)', 'U2_Right_Heatmap_Animation.gif')

# print("Creating U2 Center Side Heatmap GIF ...")
# make_heatmap_gif(all_u2_center_data, 'coolwarm',
#                  'U2 Displacement (Center Block)', 'U2_Center_Heatmap_Animation.gif')

# # ======== 相對位移 GIF（單色 bar） ========
# print("Creating Relative U2 Displacement Heatmap GIF ...")
# if all_u2_relative_data:
#     vals = np.concatenate([d['U'][~np.isnan(d['U'])] for d in all_u2_relative_data])
#     vmin, vmax = np.percentile(vals, [1, 99])

#     y_min = min(d['y_range'][0] for d in all_u2_relative_data)
#     y_max = max(d['y_range'][1] for d in all_u2_relative_data)
#     z_min = min(d['z_range'][0] for d in all_u2_relative_data)
#     z_max = max(d['z_range'][1] for d in all_u2_relative_data)

#     fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

#     # -- 畫第一幀並建立 colorbar --
#     d0 = all_u2_relative_data[0]
#     im = ax.contourf(d0['Y'], d0['Z'], d0['U'], 50, cmap='seismic', vmin=vmin, vmax=vmax)
#     cbar = fig.colorbar(im, ax=ax, label='Relative U2 (mm)')

#     def animate_rel(i):
#         d = all_u2_relative_data[i]
#         ax.clear()
#         new_im = ax.contourf(d['Y'], d['Z'], d['U'], 50, cmap='Purples', vmin=vmin, vmax=vmax)
#         # 更新 colorbar
#         cbar.update_normal(new_im)

#         ax.set(xlabel='Y (mm)', ylabel='Z (mm)',
#                title=f'Relative U2 (Right – Center) – {d["pos"]} mm\n'
#                      f'{normal_stress:.1f} MPa,  E={Spring_stiffness} MPa',
#                xlim=(y_min, y_max), ylim=(z_min, z_max), aspect='equal')
#         ax.grid(True, alpha=0.3)
#         return []

#     anim_rel = FuncAnimation(fig, animate_rel, frames=len(all_u2_relative_data), interval=500, blit=False)
#     rel_path = os.path.join(GIF_FOLDER, 'U2_Relative_Heatmap_Animation.gif')
#     anim_rel.save(rel_path, writer=PillowWriter(fps=2), dpi=150)
#     plt.close(fig)
#     print(f"Relative U2 Displacement Heatmap GIF saved: {rel_path}")

# print("\nAll GIF files saved in:", GIF_FOLDER)
# print("Processed positions:", all_positions)

In [4]:
# ---------- 通用：畫 Right / Center GIF ----------
def make_heatmap_gif(data_list, title_prefix, fname,
                     label='U2 Displacement (mm)'):
    if not data_list:
        return

    # 計算全局色階
    vals = np.concatenate([d['U'][~np.isnan(d['U'])] for d in data_list])
    vmin, vmax = np.percentile(vals, [1, 99])

    y_min = min(d['y_range'][0] for d in data_list)
    y_max = max(d['y_range'][1] for d in data_list)
    z_min = min(d['z_range'][0] for d in data_list)
    z_max = max(d['z_range'][1] for d in data_list)

    fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

    # --- 畫第一幀並建立 colorbar ---
    d0 = data_list[0]
    im = ax.contourf(d0['Y'], d0['Z'], d0['U'],
                     levels=50, cmap='Purples', vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax, label=label)

    def animate(i):
        d = data_list[i]
        ax.clear()                                      # 清主圖
        new_im = ax.contourf(d['Y'], d['Z'], d['U'],
                             levels=50, cmap='Purples',
                             vmin=vmin, vmax=vmax)
        cbar.update_normal(new_im)                      # 更新 colorbar

        ax.set(xlabel='Y (mm)', ylabel='Z (mm)',
               title=f'{title_prefix} – {d["pos"]} mm\n'
                     f'{normal_stress:.1f} MPa,  E={Spring_stiffness} MPa',
               xlim=(y_min, y_max), ylim=(z_min, z_max), aspect='equal')
        ax.grid(True, alpha=0.3)
        return []                                       # blit=False 不需回傳 artist

    anim = FuncAnimation(fig, animate, frames=len(data_list),
                         interval=500, blit=False)
    path = os.path.join(GIF_FOLDER, fname)
    anim.save(path, writer=PillowWriter(fps=2), dpi=150)
    plt.close(fig)
    print(f"{title_prefix} GIF saved: {path}")

# ---------- 相對位移 GIF ----------
def make_relative_gif(rel_list, fname='U2_Relative_Heatmap_Animation.gif'):
    if not rel_list:
        return

    vals = np.concatenate([d['U'][~np.isnan(d['U'])] for d in rel_list])
    vmin, vmax = np.percentile(vals, [1, 99])

    y_min = min(d['y_range'][0] for d in rel_list)
    y_max = max(d['y_range'][1] for d in rel_list)
    z_min = min(d['z_range'][0] for d in rel_list)
    z_max = max(d['z_range'][1] for d in rel_list)

    fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)

    d0 = rel_list[0]
    im = ax.contourf(d0['Y'], d0['Z'], d0['U'],
                     levels=50, cmap='Purples', vmin=vmin, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax, label='Relative U2 (mm)')

    def animate(i):
        d = rel_list[i]
        ax.clear()
        new_im = ax.contourf(d['Y'], d['Z'], d['U'],
                             levels=50, cmap='Purples',
                             vmin=vmin, vmax=vmax)
        cbar.update_normal(new_im)

        ax.set(xlabel='Y (mm)', ylabel='Z (mm)',
               title=f'Relative U2 (Right - Center) - {d["pos"]} mm {normal_stress:.1f} MPa,  E={Spring_stiffness} MPa',
               xlim=(y_min, y_max), ylim=(z_min, z_max), aspect='equal')
        ax.grid(True, alpha=0.3)
        return []

    anim = FuncAnimation(fig, animate, frames=len(rel_list),
                         interval=500, blit=False)
    path = os.path.join(GIF_FOLDER, fname)
    anim.save(path, writer=PillowWriter(fps=2), dpi=150)
    plt.close(fig)
    print(f"Relative U2 Displacement Heatmap GIF saved: {path}")

# ====== 產生三支 GIF ======
print("\nCreating U2 Right Side Heatmap GIF ...")
make_heatmap_gif(all_u2_right_data,
                 'U2 Displacement (Right Side)',
                 'U2_Right_Heatmap_Animation.gif')

print("Creating U2 Center Side Heatmap GIF ...")
make_heatmap_gif(all_u2_center_data,
                 'U2 Displacement (Center Block)',
                 'U2_Center_Heatmap_Animation.gif')

print("Creating Relative U2 Displacement Heatmap GIF ...")
make_relative_gif(all_u2_relative_data)


Creating U2 Right Side Heatmap GIF ...
U2 Displacement (Right Side) GIF saved: ./Disp-Ana/12.5MPa/ShearFace/GIFS/U2_Right_Heatmap_Animation.gif
Creating U2 Center Side Heatmap GIF ...
U2 Displacement (Center Block) GIF saved: ./Disp-Ana/12.5MPa/ShearFace/GIFS/U2_Center_Heatmap_Animation.gif
Creating Relative U2 Displacement Heatmap GIF ...
Relative U2 Displacement Heatmap GIF saved: ./Disp-Ana/12.5MPa/ShearFace/GIFS/U2_Relative_Heatmap_Animation.gif
